# 학교 전용 · H&M GraphSAGE seed50 M4 단독

기존 seed50 설계·L4·배치131072·300epoch 유지. 첫 실행은 처음부터 학습합니다. 다른 모델·시드와 기존 개인 checkpoint는 학습/복구하지 않습니다.

학교 계정 jungunb@u.sogang.ac.kr 사용. M3/M4를 별도 GPU에서 병렬 실행합니다. 기존 개인 M3/M4 실행은 중지되어 있어야 합니다.

매 epoch 전체 optimizer·난수 상태를 Drive에 검증 저장하며 최신+직전만 유지합니다. 정리는 이 새 실행의 더 오래된 epoch 사본에만 한정합니다. 과거 실험을 자동 삭제하지 않습니다. 두 동시 업로드까지 고려해 시작 전 8.2GB 여유를 확인합니다.

마지막 주 test만 고정300epoch에서 평가. validation·중간 test·튜닝 없음. M1·M2 완료 결과는 그대로 보존하며 이 노트북에서 다시 평가하지 않습니다.

## 1. 학교 계정·GPU·용량 확인 및 고정 코드 준비

In [ ]:
from google.colab import drive, auth
from pathlib import Path
import subprocess, sys, hashlib, json, shutil
import torch
LABEL='M4'
SOURCE_COMMIT='17aff7a9b3a6b12448fa4fbca5fedd4e6977b30b'
assert torch.cuda.is_available() and 'L4' in torch.cuda.get_device_name(0), '기존 조건과 같은 L4 GPU를 선택하세요'
drive.mount('/content/drive')
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
credentials,_=google.auth.default()
drive_api=build('drive','v3',credentials=credentials,cache_discovery=False)
info=drive_api.about().get(fields='user(emailAddress),storageQuota').execute()
assert info['user']['emailAddress']=='jungunb@u.sogang.ac.kr', '학교 계정으로 인증해야 합니다'
print('학교 계정 확인 완료. 개인 소유 파일 기준으로 용량을 계산합니다.',flush=True)
# Two arms retain two full checkpoints each; a third upload can overlap.
# Conservatively reserve both jobs' worst-case peak before either starts.
REPO=Path('/content/graphsage_school_'+LABEL.lower()+'_s50')
HELPER_PATH=Path('/content/clv_hm_graphsage_school_single.py')
if not REPO.exists():
    subprocess.run(['git','clone','--branch','feat/m2-joint-nv-lightgcn','https://github.com/jung-un/clv-m2-lightgcn-runner.git',str(REPO)],check=True)
    subprocess.run(['git','-C',str(REPO),'checkout',SOURCE_COMMIT],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()==SOURCE_COMMIT
if not HELPER_PATH.exists() or hashlib.sha256(HELPER_PATH.read_bytes()).hexdigest()!='4f7b0ad0fe80cde3d8004c21b883e78e83176019c2bae2c57436f4f974e9010f':
    from urllib.request import urlopen
    with urlopen('https://raw.githubusercontent.com/jung-un/clv-m2-lightgcn-runner/feat/m2-joint-nv-lightgcn/clv_hm_graphsage_school_single.py',timeout=30) as response:
        helper=response.read()
    assert hashlib.sha256(helper).hexdigest()=='4f7b0ad0fe80cde3d8004c21b883e78e83176019c2bae2c57436f4f974e9010f', '실행 도우미 버전 불일치: 파일은 교체하지 않았습니다'
    HELPER_PATH.write_bytes(helper)
assert hashlib.sha256(HELPER_PATH.read_bytes()).hexdigest()=='4f7b0ad0fe80cde3d8004c21b883e78e83176019c2bae2c57436f4f974e9010f'
sys.path.insert(0,str(REPO)); sys.path.insert(0,'/content')
import clv_hm_graphsage_core4_seed50 as screen
import clv_hm_graphsage_school_single as single
import importlib
single=importlib.reload(single)
assert Path(screen.__file__).resolve().parent==REPO.resolve()
DATA=Path('/content/drive/MyDrive/논문/data')
single.capacity_check(screen,drive_api,DATA)
OUT=DATA/('results_v3_hm_graphsage_seed50_school_'+LABEL.lower()+'_fresh_v1')
OUT.mkdir(parents=True,exist_ok=True)
print('학교 전용',LABEL,'seed50 — 기존 개인 M3 checkpoint를 가져오지 않습니다.',flush=True)


## 2. 입력을 로컬에 한 번 복사하고 기존 입력·분할 검증

In [ ]:
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(8*1024**2),b''): h.update(block)
    return h.hexdigest()
EXPECTED = {
 'transactions_train.parquet':'0d90884fc1bc400161e6aeba238cba94183998bedfeb4ebbf4962e17d0c8489b',
 'articles.csv':'25f21882053517f7684ea43f524225a9481a7bb9aab9cf0abfc7fe8c4576a72b'}
LOCAL = Path('/content/hm_graphsage_seed50_inputs')
LOCAL.mkdir(exist_ok=True)
for key,name in [('tx_path','transactions_train.parquet'),('item_meta_path','articles.csv')]:
    source=DATA/'raw/hm'/name
    target=LOCAL/name
    if not target.is_file() or sha256(target)!=EXPECTED[name]:
        assert source.is_file(), f'기존 학교 Drive 원본 파일 없음: {source}'
        temporary=target.with_suffix(target.suffix+'.partial')
        shutil.copyfile(source,temporary)
        assert sha256(temporary)==EXPECTED[name], f'원본 바이트 불일치: {name}'
        temporary.replace(target)
    screen.base.v3.SCHEMA['hm'][key]=str(target)
defaults=screen.configure()
ref_specs=[
 (Path(defaults.m2_reference),'clv_m2_adaptive_hm_seed49_lastweek_results.zip'),
 (Path(defaults.m4_reference),'clv_hm_m4c_lastweek_seed49_results.zip')]
refs=[]
for directory,name in ref_specs:
    archive=DATA/'references'/name
    refs.append(directory if directory.is_dir() else archive)
missing=[str(p) for p in refs if not p.exists()]
assert not missing, f'학교 Drive 참조 결과가 없습니다: {missing}. 파일 선택 위젯 대신 해당 경로에 기존 ZIP을 복사하세요.'
cfg=screen.configure(out_dir=str(OUT),m2_reference=str(refs[0]),m4_reference=str(refs[1]))
assert cfg.seeds==(50,) and cfg.epochs==300
print('입력 준비 시작: 대용량 H&M 그래프 구성에는 시간이 필요합니다.',flush=True)
if 'prepared' not in globals():
    prepared=screen.prepare(cfg)
else:
    assert prepared['cfg']==cfg, '다른 실행의 입력 재사용 금지'
checkpoint_folder=single.install(screen,drive_api,prepared,LABEL)
assert set(prepared['data']['splits'])=={'test'}
assert prepared['data'].get('loss_w') is None
print('사전점검 완료:', LABEL, '/ seed50 / 300epoch / 이 모델 하나만 실행',flush=True)
print('결과:',prepared['run_dir'])
print('재개 파일:', 'https://drive.google.com/drive/folders/'+checkpoint_folder)


## 3. M4만 학습 → 300epoch 평가 → 결과 ZIP 저장

In [ ]:
assert cfg.seeds==(50,) and LABEL in ('M3','M4')
print('학습 대상:',LABEL,'seed50 하나. 새 실행은 1epoch부터, 이 폴더에서 중단된 실행만 이어갑니다.',flush=True)
try:
    result=screen.run_arm(prepared,LABEL)
finally:
    output,status=single.export(screen,prepared,LABEL,
        '/content/hm_graphsage_'+LABEL.lower()+'_seed50_school_results.zip')
    backup=prepared['run_dir']/output.name
    shutil.copyfile(output,backup)
    assert sha256(output)==sha256(backup)
    print(json.dumps(status,ensure_ascii=False,indent=2))
    print('Drive 결과 ZIP:',backup,flush=True)


## 4. 결과 다운로드 — Drive에도 저장되어 있습니다

In [ ]:
from google.colab import files
assert status['complete'], '미완료: partial ZIP을 완료 결과로 취급하지 마세요'
files.download(str(output))
